# 00 — Data Pull
Loads all raw source files, filters to NJ where needed, and saves slim outputs to `data/01_pulls/`.

⚠️ Several raw files are **too large for GitHub** and must be downloaded separately.  
See the README for download links and exact filenames.  
If you only want to run the analysis, skip this notebook — all outputs are already in `data/01_pulls/`.

In [1]:
# ── Imports ──────────────────────────────────────────────────────────────────
import os
import subprocess
import sys
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
from pathlib import Path

# Install gdown if not already installed
try:
    import gdown
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'gdown', '-q'])
    import gdown

# ── Helper functions ──────────────────────────────────────────────────────────
def find_repo_root(start=Path().resolve()):
    """Search upward for the repo root (directory containing README.md + data/)."""
    for parent in [start] + list(start.parents):
        if (parent / "README.md").exists() and (parent / "data").exists():
            return parent
    raise FileNotFoundError("Could not find repo root — make sure README.md and data/ exist.")


def download_if_missing(dest_path, file_id):
    """Download a single file from Google Drive if it doesn't already exist locally."""
    dest_path = Path(dest_path)
    dest_path.parent.mkdir(parents=True, exist_ok=True)
    if dest_path.exists():
        print(f'  ✓ Already exists: {dest_path.name}')
    else:
        print(f'  ↓ Downloading:    {dest_path.name}')
        gdown.download(id=file_id, output=str(dest_path), quiet=False)


def download_folder_if_missing(dest_path, folder_id):
    """Download a Google Drive folder into dest_path if it doesn't already exist."""
    dest_path = Path(dest_path)
    if dest_path.exists() and any(dest_path.iterdir()):
        print(f'  ✓ Already exists: {dest_path.name}/')
    else:
        print(f'  ↓ Downloading folder into: {dest_path}/')
        dest_path.mkdir(parents=True, exist_ok=True)
        gdown.download_folder(id=folder_id, output=str(dest_path), quiet=False)


# ── Paths ─────────────────────────────────────────────────────────────────────
REPO_ROOT = find_repo_root()
DATA_RAW  = REPO_ROOT / "data" / "raw"
DATA_OUT  = REPO_ROOT / "data" / "01_pulls"
DATA_OUT.mkdir(parents=True, exist_ok=True)
os.chdir(REPO_ROOT)

print("Repo root:", REPO_ROOT)
print("Raw data: ", DATA_RAW)
print("Output:   ", DATA_OUT)

Repo root: /Users/charlottewest/Documents/QSS20-CW-Final
Raw data:  /Users/charlottewest/Documents/QSS20-CW-Final/data/raw
Output:    /Users/charlottewest/Documents/QSS20-CW-Final/data/01_pulls


## Download Large Raw Files from Google Drive
Run this cell once to download the large raw files automatically. Files that already exist locally are skipped.

## All large raw files are stored here:

https://drive.google.com/drive/folders/16chMCsDAEOjsihWfeYvnC3JGhDkNuXH9

In [2]:
# ── Google Drive file IDs ─────────────────────────────────────────────────────
# Large raw files are stored at:
# https://drive.google.com/drive/folders/16chMCsDAEOjsihWfeYvnC3JGhDkNuXH9

DRIVE_FILES = {
    DATA_RAW / 'UCMR5_All_MA_WY.txt':
        '1lKQfosj0W9cmTlQOoKB_lvyOlsllhnun',
    DATA_RAW / 'Zip_zhvi_uc_sfrcondo_tier_0.33_0.67_sm_sa_month.csv':
        '1hY3HQIsnD-whuRTLDUYAxVsEuPE-oss-',
    DATA_RAW / 'PWS_Boundaries' / 'Service_Areas_V_3_0.gpkg':
        '11IJO9lykxlsJoSodnhKfU_o7vKOzBxVJ',
}

# Shapefile components downloaded individually into data/raw/tl_2024_us_zcta520/
SHAPEFILE_DIR   = DATA_RAW / 'tl_2024_us_zcta520'
SHAPEFILE_FILES = {
    'tl_2024_us_zcta520.shp'         : '1GoFqyW3Qf23_hnis_Xp3tbQ1Xbq_em78',
    'tl_2024_us_zcta520.dbf'         : '1vA8osXGbH55HIelnT3MRGVDGOfLoko7j',
    'tl_2024_us_zcta520.shx'         : '1mMq0GVewpv9gLL8yYsw3PnZSkFud8G7t',
    'tl_2024_us_zcta520.prj'         : '1MP_XpoKMo2GGS_US4oHDPrD8GdLAnF-t',
    'tl_2024_us_zcta520.cpg'         : '1JY_QQ8oI7N3useiUzOezCa0KeO7-DP32',
    'tl_2024_us_zcta520.shp.iso.xml' : '1Pzgw0s7K7ehcs5XVQ-HuP6faMSeiykhZ',
}

# ── Run downloads ─────────────────────────────────────────────────────────────
print('Checking/downloading raw files from Google Drive...\n')

for dest, fid in DRIVE_FILES.items():
    download_if_missing(dest, fid)

SHAPEFILE_DIR.mkdir(parents=True, exist_ok=True)
for filename, fid in SHAPEFILE_FILES.items():
    download_if_missing(SHAPEFILE_DIR / filename, fid)

print('\nAll files ready.')

Checking/downloading raw files from Google Drive...

  ✓ Already exists: UCMR5_All_MA_WY.txt
  ✓ Already exists: Zip_zhvi_uc_sfrcondo_tier_0.33_0.67_sm_sa_month.csv
  ✓ Already exists: Service_Areas_V_3_0.gpkg
  ↓ Downloading:    tl_2024_us_zcta520.shp


Downloading...
From (original): https://drive.google.com/uc?id=1GoFqyW3Qf23_hnis_Xp3tbQ1Xbq_em78
From (redirected): https://drive.google.com/uc?id=1GoFqyW3Qf23_hnis_Xp3tbQ1Xbq_em78&confirm=t&uuid=eeca5586-cfae-4c52-bff4-6409396579e0
To: /Users/charlottewest/Documents/QSS20-CW-Final/data/raw/tl_2024_us_zcta520/tl_2024_us_zcta520.shp
100%|████████████████████████████████████████| 822M/822M [00:22<00:00, 36.8MB/s]


  ↓ Downloading:    tl_2024_us_zcta520.dbf


Downloading...
From: https://drive.google.com/uc?id=1vA8osXGbH55HIelnT3MRGVDGOfLoko7j
To: /Users/charlottewest/Documents/QSS20-CW-Final/data/raw/tl_2024_us_zcta520/tl_2024_us_zcta520.dbf
100%|██████████████████████████████████████| 2.84M/2.84M [00:00<00:00, 22.0MB/s]


  ↓ Downloading:    tl_2024_us_zcta520.shx


...

  ↓ Downloading:    tl_2024_us_zcta520.cpg


Downloading...
From: https://drive.google.com/uc?id=1JY_QQ8oI7N3useiUzOezCa0KeO7-DP32
To: /Users/charlottewest/Documents/QSS20-CW-Final/data/raw/tl_2024_us_zcta520/tl_2024_us_zcta520.cpg
100%|████████████████████████████████████████| 5.00/5.00 [00:00<00:00, 10.9kB/s]


  ↓ Downloading:    tl_2024_us_zcta520.shp.iso.xml


Downloading...
From (original): https://drive.google.com/uc?id=1Pzgw0s7K7ehcs5XVQ-HuP6faMSeiykhZ
From (redirected): https://drive.google.com/uc?id=1Pzgw0s7K7ehcs5XVQ-HuP6faMSeiykhZ&confirm=t&uuid=c8c573d2-73be-47af-9e08-f26bc22dad27
To: /Users/charlottewest/Documents/QSS20-CW-Final/data/raw/tl_2024_us_zcta520/tl_2024_us_zcta520.shp.iso.xml
100%|██████████████████████████████████████| 50.7k/50.7k [00:00<00:00, 2.82MB/s]


All files ready.


## UCMR5 Dataset

In [12]:
# Slim down the UCMR dataset so it is small enough for GitHub
# Raw source: https://www.epa.gov/dwucmr/occurrence-data-unregulated-contaminant-monitoring-rule
# Download "UCMR5_All_MA_WY.txt" and place it in data/raw/
ucmr_full = pd.read_csv(
    DATA_RAW / "UCMR5_All_MA_WY.txt",
    sep      = '\t',
    encoding = 'cp1252',
    dtype    = {'PWSID': str, 'ZIPCode': str}
)

print(f'Full file rows:    {len(ucmr_full)}')
print(f'Full file columns: {ucmr_full.columns.tolist()}')

#keeping only the relevant columns
cols_to_keep = [
    'PWSID',                   # join key to ZIP codes file
    'PWSName',                 # water system name
    'State',                   # for filtering to NJ
    'Size',                    # S or L (small/large system)
    'FacilityWaterType',       # surface water vs groundwater
    'Contaminant',             # PFAS type
    'AnalyticalResultValue',   # the actual concentration
    'AnalyticalResultsSign',   # < (non-detect) or = (detected)
    'MRL',                     # minimum reporting level
    'Units',                   # µg/L
    'CollectionDate',          # sample date
    'Region',                  # EPA region
]

ucmr_slim = ucmr_full[cols_to_keep].copy()

#Filter to NJ only
ucmr_nj = ucmr_slim[
    (ucmr_slim['State'] == 'NJ') &
    (ucmr_slim['PWSID'].str.startswith('NJ'))
].copy()

print(f'\nBefore: {len(ucmr_full)} rows, {len(ucmr_full.columns)} columns')
print(f'After:  {len(ucmr_nj)} rows, {len(ucmr_nj.columns)} columns')

Full file rows:    1150728
Full file columns: ['PWSID', 'PWSName', 'Size', 'FacilityID', 'FacilityName', 'FacilityWaterType', 'SamplePointID', 'SamplePointName', 'SamplePointType', 'AssociatedFacilityID', 'AssociatedSamplePointID', 'CollectionDate', 'SampleID', 'Contaminant', 'MRL', 'Units', 'MethodID', 'AnalyticalResultsSign', 'AnalyticalResultValue', 'SampleEventCode', 'MonitoringRequirement', 'Region', 'State', 'UCMR1SampleType']

Before: 1150728 rows, 24 columns
After:  56542 rows, 12 columns


## Zipcode Data

In [8]:
# Load ZIP Code Data
# Raw source: same UCMR5 download — place "UCMR5_ZIPCodes.txt" in data/raw/
nj_zc = pd.read_csv(
    DATA_RAW / "UCMR5_ZIPCodes.txt",
    sep="\t", encoding="cp1252",
    dtype={'PWSID': str, 'ZIPCODE': str}
)

nj_zc

,PWSID,ZIPCODE
0,010106001,06338
1,010109005,06382
2,020000005,13655
3,020000008,14070
4,020000008,14081
...,...,...
31102,WY5601569,83116
31103,WY5680074,82190
31104,WY5680085,82190
31105,WY5680095,83012


## Census Data

In [9]:
# Load Census Data
# Raw source: Census ACS DP05 table for NJ ZCTAs — place "ACSDP5Y2024.DP05-Data.csv" in data/raw/,
# and Census ACS S1901 table for NJ "ACSDP5Y2024.DP05-Data.csv"
census_race_df = pd.read_csv(
    DATA_RAW / "ACSDP5Y2024.DP05-Data.csv",
    header=0,
    skiprows=[1],
    dtype={'GEO_ID': str}
)

census_income_df = pd.read_csv(
    DATA_RAW / "ACSST5Y2024.S1901-Data.csv",
    header = 0,
    skiprows = [1],
    dtype={'GEO_ID': str}
)

census_income_df

,GEO_ID,NAME,S1901_C01_001E,S1901_C01_001M,S1901_C01_002E,S1901_C01_002M,S1901_C01_003E,S1901_C01_003M,S1901_C01_004E,S1901_C01_004M,...,S1901_C04_012M,S1901_C04_013E,S1901_C04_013M,S1901_C04_014E,S1901_C04_014M,S1901_C04_015E,S1901_C04_015M,S1901_C04_016E,S1901_C04_016M,Unnamed: 130
0,0400000US34,New Jersey,3507701,5116,4.1,0.1,2.7,0.1,4.8,0.1,...,554,83840,829,(X),(X),(X),(X),33.5,(X),NaN
1,860Z200US07001,ZCTA5 07001,5615,497,4.0,3.0,1.2,1.1,3.2,2.5,...,34319,82265,14811,(X),(X),(X),(X),50.1,(X),NaN
2,860Z200US07002,ZCTA5 07002,28951,793,4.9,1.1,4.7,1.2,6.0,1.3,...,4100,77800,11912,(X),(X),(X),(X),29.9,(X),NaN
3,860Z200US07003,ZCTA5 07003,21215,706,2.7,0.9,2.0,0.8,4.9,1.3,...,1825,108719,26584,(X),(X),(X),(X),26.1,(X),NaN
4,860Z200US07004,ZCTA5 07004,3039,272,6.3,5.7,0.0,1.5,2.5,2.9,...,64526,67047,17078,(X),(X),(X),(X),33.1,(X),NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
594,860Z200US08889,ZCTA5 08889,4333,283,3.2,2.9,1.4,1.2,1.9,1.0,...,13506,96307,24568,(X),(X),(X),(X),17.5,(X),NaN
595,860Z200US08890,ZCTA5 08890,0,14,-,**,-,**,-,**,...,**,-,**,(X),(X),(X),(X),-,(X),NaN
596,860Z200US08901,ZCTA5 08901,15676,808,12.3,2.3,5.2,2.3,10.1,2.3,...,10334,70997,9964,(X),(X),(X),(X),36.1,(X),NaN
597,860Z200US08902,ZCTA5 08902,15398,545,2.9,1.1,1.6,0.9,3.8,1.2,...,11299,95092,9765,(X),(X),(X),(X),22.3,(X),NaN


## Public Water System Boundaries

In [10]:
# ⚠️ HARDCODED RAW FILES — too large for GitHub, must be downloaded separately.
# Service_Areas_V_3_0.gpkg: downloaded automatically from Google Drive (see cell above)
# Tracts_V_3_0.csv: committed directly to data/raw/ on GitHub

# --- Service area polygons (CWS layer) ---
print("Loading PWS service area polygons...")
pws_full = gpd.read_file(
    DATA_RAW / "PWS_Boundaries" / "Service_Areas_V_3_0.gpkg",
    layer = 'CWS'
)
print(f'  Full file rows: {len(pws_full)}')

pws_nj = pws_full[pws_full['PWSID'].str.startswith('NJ')].copy()
print(f'  NJ only rows:   {len(pws_nj)}')

# --- Census tract crosswalk (committed to data/raw/ directly) ---
print("Loading PWS tract crosswalk...")
pws_tracts_full = pd.read_csv(
    DATA_RAW / "Tracts_V_3_0.csv",
    dtype = {'GEOID20': str, 'PWSID': str}
)
print(f'  Full crosswalk rows: {len(pws_tracts_full)}')

pws_tracts_nj = pws_tracts_full[pws_tracts_full['PWSID'].str.startswith('NJ')].copy()
print(f'  NJ only rows:        {len(pws_tracts_nj)}')
pws_tracts_nj.head()

Loading PWS service area polygons...
  Full file rows: 44656
  NJ only rows:   558
Loading PWS tract crosswalk...
  Full crosswalk rows: 165121
  NJ only rows:        3833


,GEOID20,PWSID,Tract_Km,Tract_I_Km,Area_Weight,Pop20_AW,Tract_Buildings,Tract_O_Buildings,Bldg_Weight,Pop20_BW
89944,34001000200,NJ0102001,0.523371,0.487665,0.931777,2699,597.0,596,0.998325,2892.0
89945,34001000200,NJ0122001,0.523371,0.000003,0.000006,0,597.0,0,0.000000,0.0
89946,34001000300,NJ0102001,0.365553,0.341909,0.935319,3767,404.0,403,0.997525,4017.0
89947,34001000400,NJ0102001,0.676905,0.597791,0.883124,2864,276.0,270,0.978261,3172.0
89948,34001000500,NJ0102001,0.273704,0.270606,0.988680,2978,394.0,394,1.000000,3012.0


## ECHO PFAS Handling Sectors

In [9]:
# Load ECHO PFAS-Handling Sectors data for NJ
# Already filtered to NJ — small enough to commit directly to data/01_pulls/
# Raw source: EPA ECHO (https://echo.epa.gov)
# File is in data/raw/ as the downloaded xlsx

echo_pfas_df = pd.read_excel(
    DATA_RAW / '2217deab-2b3d-4828-bbd2-757bcbe06e2c.xlsx',
    dtype={'FAC_FIPS_CODE': str}
)

print(f'Rows loaded:    {len(echo_pfas_df)}')
print(f'Industry types:\n{echo_pfas_df["Industry"].value_counts()}')
echo_pfas_df

Rows loaded:    3757
Industry types:
Industry
Chemical Mfg                662
Waste Management            416
Airports                    364
Plastics and Resins         348
Electronics Industry        329
Metal Coating               309
Petroleum                   199
Paints and Coatings         197
Metal Machinery Mfg         189
Printing                    183
Cleaning Product Mfg        151
Textiles and Leather        115
Paper Mills and Products     82
National Defense             74
Glass Products               54
Industrial Gas               30
Fire Training                27
Cement Mfg                    7
Furniture and Carpet          5
Consumer Products             5
Airports (Part 139)           4
Mining and Refining           4
Fire Protection               2
Oil and Gas                   1
Name: count, dtype: int64


,Facility,Region,State,State (Other),City,Status,Industry,ECHO Facility Report,FAC_PERCENT_MINORITY,FAC_DERIVED_TRIBES,...,SDWA_IDS,SDWA_SYSTEM_TYPES,SDWA_COMPLIANCE_STATUS,SDWA_SNC_FLAG,TRI_IDS,TRI_RELEASES_TRANSFERS,TRI_ON_SITE_RELEASES,TRI_OFF_SITE_TRANSFERS,TRI_REPORTER,FAC_IMP_WATER_FLG
0,FLY-N-D LANDING STRIP,2,NJ,NaN,1,Unknown,Airports,https://echo.epa.gov/detailed-facility-report?...,4.771,-,...,-,-,-,N,-,-,-,-,-,-
1,WEISS FARM,2,NJ,NaN,ALLAMUCHY,Unknown,Airports,https://echo.epa.gov/detailed-facility-report?...,8.972,-,...,-,-,-,N,-,-,-,-,-,-
2,ALLOWAY AIRFIELD,2,NJ,NaN,ALLOWAY,Unknown,Airports,https://echo.epa.gov/detailed-facility-report?...,13.559,-,...,-,-,-,N,-,-,-,-,-,-
3,TRINCA,2,NJ,NaN,ANDOVER,Unknown,Airports,https://echo.epa.gov/detailed-facility-report?...,8.77,-,...,-,-,-,N,-,-,-,-,-,-
4,AEROFLEX-ANDOVER,2,NJ,NaN,ANDOVER,Unknown,Airports,https://echo.epa.gov/detailed-facility-report?...,11.509,-,...,-,-,-,N,-,-,-,-,-,-
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3752,CAPE MAY COUNTY MUA SECURE LANDFILL,2,NJ,NaN,WOODBINE,Unknown,Waste Management,https://echo.epa.gov/detailed-facility-report?...,-,-,...,-,-,-,N,-,-,-,-,-,-
3753,WOODBRIDGE TWP MUA WOODBRIDGE AVENUE PUMP STATION,2,NJ,NaN,WOODBRIDGE,Active,Waste Management,https://echo.epa.gov/detailed-facility-report?...,58.944,-,...,-,-,-,N,-,-,-,-,-,-
3754,WOODBRIDGE TWP LANDFILL,2,NJ,NaN,WOODBRIDGE,Active,Waste Management,https://echo.epa.gov/detailed-facility-report?...,61.527,-,...,-,-,-,N,-,-,-,-,-,-
3755,MAC SLF INC,2,NJ,NaN,WOODBURY,Inactive,Waste Management,https://echo.epa.gov/detailed-facility-report?...,17.236,-,...,-,-,-,N,-,-,-,-,-,-


## Zillow Housing Data

In [11]:
# Using December 2024 home values to align with:
#   - UCMR5 collection period (Jan 2023 – Dec 2025)
#   - Census ACS 5-Year 2024 (covers 2020–2024)
ZHVI_DATE = '2024-12-31'

print("Loading Zillow ZHVI data...")
zhvi_full = pd.read_csv(
    DATA_RAW / 'Zip_zhvi_uc_sfrcondo_tier_0.33_0.67_sm_sa_month.csv',
    dtype={'RegionName': str}
)
print(f'  Full file rows: {len(zhvi_full)}')

zhvi_nj = zhvi_full[zhvi_full['State'] == 'NJ'].copy()
zhvi_nj = zhvi_nj[['RegionName', 'State', 'City', 'Metro', 'CountyName', ZHVI_DATE]].copy()
zhvi_nj.columns = ['ZIPCODE', 'State', 'City', 'Metro', 'CountyName', 'zhvi']
zhvi_nj['ZIPCODE'] = zhvi_nj['ZIPCODE'].str.zfill(5)
zhvi_nj['zhvi'] = pd.to_numeric(zhvi_nj['zhvi'], errors='coerce')

print(f'  NJ only rows:        {len(zhvi_nj)}')
print(f'  Home value date:     {ZHVI_DATE}')
print(f'  Missing ZHVI values: {zhvi_nj["zhvi"].isna().sum()}')
zhvi_nj.head()

Loading Zillow ZHVI data...
  Full file rows: 26276
  NJ only rows:        547
  Home value date:     2024-12-31
  Missing ZHVI values: 0


,ZIPCODE,State,City,Metro,CountyName,zhvi
1,08701,NJ,Lakewood,"New York-Newark-Jersey City, NY-NJ-PA",Ocean County,528001.198927
209,07055,NJ,Passaic,"New York-Newark-Jersey City, NY-NJ-PA",Passaic County,528951.812764
218,07305,NJ,Jersey City,"New York-Newark-Jersey City, NY-NJ-PA",Hudson County,549833.496972
263,07087,NJ,Union City,"New York-Newark-Jersey City, NY-NJ-PA",Hudson County,506978.414689
337,08753,NJ,Toms River,"New York-Newark-Jersey City, NY-NJ-PA",Ocean County,517253.823364


## NJ Zipcode Shape files

In [14]:
print("Loading NJ ZIP code shapefiles...")
nj_shapefile = gpd.read_file(DATA_RAW / "tl_2024_us_zcta520")
print(f'  Full file rows: {len(nj_shapefile)}')

nj_shapefile = nj_shapefile[
    nj_shapefile['ZCTA5CE20'].str.startswith('07') |
    nj_shapefile['ZCTA5CE20'].str.startswith('08')
].copy()
nj_shapefile_slim = nj_shapefile.rename(columns={'ZCTA5CE20': 'ZIPCODE'})
print(f'  NJ only rows:   {len(nj_shapefile_slim)}')
nj_shapefile_slim.head()

Loading NJ ZIP code shapefiles...
  Full file rows: 33791
  NJ only rows:   598


,ZIPCODE,GEOID20,GEOIDFQ20,CLASSFP20,MTFCC20,FUNCSTAT20,ALAND20,AWATER20,INTPTLAT20,INTPTLON20,geometry
21637,07066,07066,860Z200US07066,B5,G6350,S,11337641,466241,+40.6206502,-074.3098621,"POLYGON ((-74.35868 40.60409, -74.35863 40.604..."
21638,07608,07608,860Z200US07608,B5,G6350,S,4431934,26306,+40.8496088,-074.0618930,"POLYGON ((-74.07352 40.84149, -74.07202 40.842..."
21639,08723,08723,860Z200US08723,B5,G6350,S,30976323,9466538,+40.0385836,-074.1116001,"POLYGON ((-74.16044 40.05879, -74.16037 40.059..."
21640,08088,08088,860Z200US08088,B5,G6350,S,389829618,4466960,+39.8373145,-074.6952670,"POLYGON ((-74.81451 39.78551, -74.81336 39.788..."
21641,08008,08008,860Z200US08008,B5,G6350,S,27995232,88485185,+39.6235376,-074.2060888,"POLYGON ((-74.3425 39.56545, -74.34226 39.5654..."


## Saving to Github raw folder

In [13]:
# Save outputs for next notebook
DATA_OUT.mkdir(parents=True, exist_ok=True)

nj_zc.to_csv(DATA_OUT / "zipcodes_raw.csv", index=False)
census_race_df.to_csv(DATA_OUT / "census_race_raw.csv", index=False)
census_income_df.to_csv(DATA_OUT / "census_income_raw.csv", index=False)
pws_tracts_nj.to_csv(DATA_OUT / "pws_tract_crosswalk_nj.csv", index=False)
pws_nj.to_file(DATA_OUT / "pws_boundaries_nj.geojson", driver='GeoJSON')
echo_pfas_df.to_csv(DATA_OUT / "echo_pfas_facilities_nj.csv", index=False)
zhvi_nj.to_csv(DATA_OUT / "zhvi_nj.csv", index=False)

print("Saved:"
      "\n  zipcodes_raw.csv"
      "\n  census_race_raw.csv"
      "\n  census_income_raw.csv"
      "\n  pws_tract_crosswalk_nj.csv"
      "\n  pws_boundaries_nj.geojson"
      "\n  echo_pfas_facilities_nj.csv"
      "\n  zhvi_nj.csv")

Saved:
  zipcodes_raw.csv
  census_race_raw.csv
  census_income_raw.csv
  pws_tract_crosswalk_nj.csv
  pws_boundaries_nj.geojson
  echo_pfas_facilities_nj.csv
  zhvi_nj.csv


In [15]:
ucmr_nj.to_csv(DATA_OUT / "ucmr5_nj_slim.csv", index=False)
nj_shapefile_slim.to_file(
    DATA_OUT / "nj_zcta_2024.geojson",
    driver='GeoJSON')